# Medicare+ — Prescription Handwriting OCR

## Data Preparation

**Google Colab Requirements:** Runtime > Change runtime type > GPU (T4) is recommended.

### Dataset

We use [`mamun1113/doctors-handwritten-prescription-bd-dataset`](https://www.kaggle.com/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset) — ~4,700 cropped images of single drug names handwritten by doctors, with the printed name as label.

## 1. Install dependencies

In [ ]:
!pip install -q kaggle pandas scikit-learn matplotlib sentencepiece pillow 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'evaluate>=0.4' 'jiwer>=3 '

In [ ]:
from google.colab import files, drive
from pathlib import Path
import pandas as pd
import os
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, Image as HFImage

## 2. Mount Google Drive

In [ ]:
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/medicare_plus_ocr'
DATA_ROOT = f'{PROJECT_ROOT}/data'
os.makedirs(DATA_ROOT, exist_ok=True)

## 3. Authenticate to Kaggle Account

In [ ]:
os.makedirs('/root/.kaggle', exist_ok=True)

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, '/root/.kaggle/kaggle.json')

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials set up successfully.')

## 4. Import the Dataset

In [ ]:
KAGGLE_DATASET = 'mamun1113/doctors-handwritten-prescription-bd-dataset'
RAW_DIR = f'{DATA_ROOT}/raw'
os.makedirs(RAW_DIR, exist_ok=True)

if not os.listdir(RAW_DIR):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DIR} --unzip
else:
    print('Dataset already downloaded — skipping.')

!ls -lah {RAW_DIR} | head -20

## 5. Discover the dataset structure

Different versions of the Kaggle dataset organize files slightly differently (sometimes `Training/Word/<class>/*.png`, sometimes a CSV index). Let's auto-detect.

In [ ]:
raw = Path(RAW_DIR)

csv_files = list(raw.rglob('*.csv'))
image_files = [p for p in raw.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]

print(f'CSV files found: {len(csv_files)}')
print(f'\nImage files found: {len(image_files)}')

## 6. Load Dataset

In [ ]:
records = []

def _normalize_label(s: str) -> str:
    return str(s).strip().lower()

def _load_files(csv_file: list, image_file: list):
    csv_loaded = False

    for csv_path in csv_file:
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            raise FileNotFoundError(f'File Not Found Error: {csv_path}')

        cols_lower = {c.lower(): c for c in df.columns}
        img_col = next((cols_lower[k] for k in cols_lower if 'image' in k or 'file' in k or 'name' in k), None)
        label_col = next((cols_lower[k] for k in cols_lower if 'medicine' in k or 'label' in k or 'word' in k or 'text' in k), None)

        if img_col and label_col and img_col != label_col:
            print(f'Using CSV: {csv_path}\n  image col: {img_col}\n  label col: {label_col}')
            path_lookup = {p.name: p for p in image_file}

            for _, row in df.iterrows():
                img_name = str(row[img_col]).strip()
                label = _normalize_label(row[label_col])
                p = path_lookup.get(img_name) or path_lookup.get(img_name + '.png') or path_lookup.get(img_name + '.jpg')
                if p is not None and label:
                    records.append({'image_path': str(p), 'label': label})
            csv_loaded = True
            break

    if not csv_loaded:
        print('No usable CSV — falling back to folder names for labels.')
        for p in image_file:
            records.append({'image_path': str(p), 'label': _normalize_label(p.parent.name)})


def _display_dataset_info(local_df: pd.DataFrame):
    print(f"Total samples: {len(local_df)}")
    print(f'Unique labels: {local_df["label"].nunique()}')
    local_df.head()
    return local_df

df_all = (
    pd.DataFrame(records).drop_duplicates(subset=["image_path"]).reset_index(drop=True)
)

_load_files(csv_files, image_files)
_display_dataset_info(df_all)

## 7. Dataset Exploration

In [ ]:
def _display_dataset_explore(local_df: pd.DataFrame):
    label_counts = local_df['label'].value_counts()

    print('Top 20 most common drug names:')
    print(local_df['label'].value_counts().head(20))

    print(f'\nLabels with only 1 sample: {(label_counts == 1).sum()}')
    print(f'Labels with >= 5 samples: {(label_counts >= 5).sum()}')

    sample = local_df.sample(min(12, len(local_df)), random_state=42).reset_index(drop=True)
    fig, axes = plt.subplots(3, 4, figsize=(14, 8))

    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(row["label"], fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

_display_dataset_explore(df_all)

## 8. Dataframe Train / Validation / Test split (8:1:1 Ratio)

In [ ]:
def _split_dataset(local_df: pd.DataFrame, test_size=0.2, random_state=42):
    counts = local_df["label"].value_counts()
    common = local_df[local_df["label"].isin(counts[counts >= 2].index)].copy()
    rare = local_df[local_df["label"].isin(counts[counts < 2].index)].copy()

    train_data_common, test_common = train_test_split(
        common,
        test_size=0.10,
        random_state=random_state,
        stratify=common["label"] if common["label"].nunique() > 1 else None,
    )
    train_common, val_common = train_test_split(
        train_data_common,
        test_size=0.111,
        random_state=random_state,
        stratify=(
            train_data_common["label"] if train_data_common["label"].nunique() > 1 else None
        ),
    )

    train_df = (
        pd.concat([train_common, rare], ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    val_df = val_common.reset_index(drop=True)
    test_df = test_common.reset_index(drop=True)

    print(f"Train Data: {len(train_df)}")
    print(f"Validation Data: {len(val_df)}")
    print(f"Test Data: {len(test_df)}")

    return [train_df, val_df, test_df]

_split_dataset(df_all)

## 9. Save as DatasetDict

In [ ]:
def to_hf(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[['image_path', 'label']].rename(columns={'image_path': 'image'}))
    return ds.cast_column('image', HFImage())

df = _split_dataset(df_all)

def _prepare_and_save_datasets(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    splits = DatasetDict(
        {
            "train": to_hf(train_df),
            "validation": to_hf(val_df),
            "test": to_hf(test_df),
        }
    )

    PREPARED_DIR = f"{DATA_ROOT}/prepared"
    splits.save_to_disk(PREPARED_DIR)
    print(f"Saved DatasetDict to: {PREPARED_DIR}")
    print(splits)

_prepare_and_save_datasets(df[0], df[1], df[2])

# Complete Preprocessing Phase

## 10. Initialize the Training Libraries

In [ ]:
import torch
from datasets import load_from_disk
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
import shutil

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 11. Mount Drive & Load Prepared Dataset

In [ ]:
# PROJECT_ROOT = "/content/drive/MyDrive/medicare_plus_ocr"
PREPARED_DIR = f"{PROJECT_ROOT}/data/prepared"
MODEL_OUT_DIR = f"{PROJECT_ROOT}/models/trocr-prescription"
LOG_DIR = f"{PROJECT_ROOT}/logs"

BASE_MODEL = "microsoft/trocr-base-handwritten"
MAX_TARGET_LENGTH = 32

NUM_EPOCHS = 10
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 5e-5

os.makedirs(MODEL_OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

splits = load_from_disk(PREPARED_DIR)
print(splits)

## 12. Load TrOCR processor + model

In [ ]:
processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)

def _prepare_processor_and_model(max_tar_len: int):

    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.config.vocab_size = model.config.decoder.vocab_size

    model.config.max_length = max_tar_len
    model.config.num_beams = 4
    model.config.early_stopping = True
    model.config.no_repeat_ngram_size = 3
    model.config.length_penalty = 2.0

_prepare_processor_and_model(MAX_TARGET_LENGTH)

## 13. Dataset Wrapper

In [ ]:
class TrOCRDataset(Dataset):
    def __init__(self, hf_split, processor, max_target_length=MAX_TARGET_LENGTH):
        self.split = hf_split
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        row = self.split[idx]
        image = row["image"].convert("RGB")
        text = row["label"]
        pixel_values = self.processor(image, return_tensors="pt").pixel_values[0]
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
        ).input_ids
        labels = [
            t if t != self.processor.tokenizer.pad_token_id else -100 for t in labels
        ]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


train_ds = TrOCRDataset(splits["train"], processor)
val_ds = TrOCRDataset(splits["validation"], processor)

print(f"train samples: {len(train_ds)}")
print(f"val   samples: {len(val_ds)}")

## 14. CER Metric for Evaluation

In [ ]:
cer_metric = evaluate.load("cer")

def compute_metrics(eval_pred, processor):
    pred_ids, label_ids = eval_pred
    label_ids_clean = [[t for t in seq if t != -100] for seq in label_ids]
    label_str = processor.batch_decode(label_ids_clean, skip_special_tokens=True)
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

## 15. Configure Training

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=f"{LOG_DIR}/checkpoints",
    overwrite_output_dir=True,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    fp16=torch.cuda.is_available(),
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=200,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

## 16. Training Process

In [ ]:
trainer.train()

## 17. Save Final Model to Drive

In [ ]:
trainer.save_model(MODEL_OUT_DIR)
processor.save_pretrained(MODEL_OUT_DIR)
print(f'Model saved to: {MODEL_OUT_DIR}')
!ls -lah {MODEL_OUT_DIR}

## 18. Zip the Model and Offer a Download Link

In [ ]:
ZIP_PATH = "/content/trocr-prescription.zip"
shutil.make_archive(ZIP_PATH.replace(".zip", ""), "zip", MODEL_OUT_DIR)
print(f"Zip size: {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB")
files.download(ZIP_PATH)

# Complete Training Phase

## 19. Install the Evaluation Libraries

In [ ]:
import os, torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from datasets import load_from_disk
from PIL import Image as PILImage
import pandas as pd
from tqdm.auto import tqdm
import evaluate
import matplotlib.pyplot as plt
import random
import json

## 20. Mount Drive, load model & test split

In [ ]:
MODEL_DIR = f"{PROJECT_ROOT}/models/trocr-prescription"
PREPARED_DIR = f"{PROJECT_ROOT}/data/prepared"
BASE_MODEL = "microsoft/trocr-base-handwritten"

BATCH_SIZE = 16
MAX_LEN = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR).to(device).eval()

splits = load_from_disk(PREPARED_DIR)
test_split = splits["test"]
print(f"Test samples: {len(test_split)}")

## 21. Generate Predictions on the Test Set

In [ ]:
def predict_batch(images, num_beams=4, num_return_sequences=1):
    pixel_values = processor(images, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=MAX_LEN,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            early_stopping=True,
        )
    texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    return texts


preds, refs = [], []
for i in tqdm(range(0, len(test_split), BATCH_SIZE)):
    rows = test_split[i : i + BATCH_SIZE]
    images = [img.convert("RGB") for img in rows["image"]]
    labels = [str(l).strip().lower() for l in rows["label"]]
    out = predict_batch(images, num_beams=4, num_return_sequences=1)
    preds.extend([t.strip().lower() for t in out])
    refs.extend(labels)

results_df = pd.DataFrame({"reference": refs, "prediction": preds})
results_df.head(10)

## 22. CER and WER on Test Set

In [ ]:
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

cer = cer_metric.compute(predictions=preds, references=refs)
wer = wer_metric.compute(predictions=preds, references=refs)
print(f"Test CER: {cer:.4f}  (lower is better)")
print(f"Test WER: {wer:.4f}")

## 23. Exact-Match Accuracy

In [ ]:
exact = (results_df["reference"] == results_df["prediction"]).mean()
print(
    f"Exact-match accuracy: {exact:.4f}  ({int(exact * len(results_df))} / {len(results_df)})"
)

## 24. Top-K Accuracy via Beam Search

In [ ]:
K = 5
topk_hits = 0
topk_preds_all = []

for i in tqdm(range(0, len(test_split), BATCH_SIZE)):
    rows = test_split[i : i + BATCH_SIZE]
    images = [img.convert("RGB") for img in rows["image"]]
    labels = [str(l).strip().lower() for l in rows["label"]]
    out = predict_batch(images, num_beams=K, num_return_sequences=K)
    out = [
        [t.strip().lower() for t in out[j * K : (j + 1) * K]]
        for j in range(len(images))
    ]
    for label, candidates in zip(labels, out):
        if label in candidates:
            topk_hits += 1
        topk_preds_all.append(candidates)

print(f"Top-{K} accuracy: {topk_hits / len(test_split):.4f}")

## 25. Most common errors

In [ ]:
errors_df = results_df[results_df["reference"] != results_df["prediction"]].copy()
errors_df["pair"] = errors_df["reference"] + "  →  " + errors_df["prediction"]
print(f"Total errors: {len(errors_df)} / {len(results_df)}")
print("\nTop 20 most frequent confusion pairs:")
print(errors_df["pair"].value_counts().head(20).to_string())

## 26. Side-by-Side Visualization

In [ ]:
N_SHOW = 12

sample_idx = random.sample(range(len(test_split)), N_SHOW)
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for ax, idx in zip(axes.ravel(), sample_idx):
    row = test_split[idx]
    img = row["image"].convert("RGB")
    label = str(row["label"]).strip().lower()
    pred = preds[idx]
    color = "green" if pred == label else "red"
    ax.imshow(img)
    ax.set_title(f"true: {label}\npred: {pred}", fontsize=9, color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 27. Before vs After — Baseline TrOCR Comparison

In [ ]:
base_processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
base_model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL).to(device).eval()


def predict_with(p, m, images):
    pixel_values = p(images, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        ids = m.generate(
            pixel_values, max_length=MAX_LEN, num_beams=4, early_stopping=True
        )
    return [t.strip().lower() for t in p.batch_decode(ids, skip_special_tokens=True)]


SAMPLE_N = min(500, len(test_split))
sample_idx = random.sample(range(len(test_split)), SAMPLE_N)

base_preds, ft_preds, gold = [], [], []
for i in tqdm(range(0, SAMPLE_N, BATCH_SIZE)):
    chunk = sample_idx[i : i + BATCH_SIZE]
    images = [test_split[j]["image"].convert("RGB") for j in chunk]
    labels = [str(test_split[j]["label"]).strip().lower() for j in chunk]
    base_preds.extend(predict_with(base_processor, base_model, images))
    ft_preds.extend(predict_with(processor, model, images))
    gold.extend(labels)

base_cer = cer_metric.compute(predictions=base_preds, references=gold)
ft_cer = cer_metric.compute(predictions=ft_preds, references=gold)
base_exact = sum(p == g for p, g in zip(base_preds, gold)) / len(gold)
ft_exact = sum(p == g for p, g in zip(ft_preds, gold)) / len(gold)

print(f'{"Metric":<20}{"Baseline":<15}{"Fine-tuned":<15}{"Delta":<10}')
print(f'{"-"*60}')
print(
    f'{"CER (lower=better)":<20}{base_cer:<15.4f}{ft_cer:<15.4f}{ft_cer - base_cer:+.4f}'
)
print(
    f'{"Exact match (acc)":<20}{base_exact:<15.4f}{ft_exact:<15.4f}{ft_exact - base_exact:+.4f}'
)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(["Baseline", "Fine-tuned"], [base_cer, ft_cer])
ax[0].set_title("CER (lower is better)")
ax[0].set_ylabel("CER")

ax[1].bar(["Baseline", "Fine-tuned"], [base_exact, ft_exact])
ax[1].set_title("Exact-match accuracy")
ax[1].set_ylabel("Accuracy")
ax[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 28. Save Evaluation Report

In [ ]:
REPORT_DIR = f"{PROJECT_ROOT}/reports"
os.makedirs(REPORT_DIR, exist_ok=True)

results_df.to_csv(f"{REPORT_DIR}/test_predictions.csv", index=False)
summary = {
    "test_cer": cer,
    "test_wer": wer,
    "test_exact_match": exact,
    f"top_{K}_accuracy": topk_hits / len(test_split),
    "baseline_sample_cer": base_cer,
    "finetuned_sample_cer": ft_cer,
    "baseline_sample_exact": base_exact,
    "finetuned_sample_exact": ft_exact,
    "test_samples": len(test_split),
    "comparison_sample_size": SAMPLE_N,
}
with open(f"{REPORT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nSaved to: {REPORT_DIR}")